# Roman to Integer — Nhập môn dễ hiểu

Bài toán LeetCode:

```python
class Solution:
    def romanToInt(self, s: str) -> int:
        ...
```

## Mục tiêu của notebook

Sau notebook này, bạn sẽ hiểu:

- Số La Mã được biểu diễn như thế nào.
- Vì sao có những trường hợp phải **trừ** như `IV = 4`.
- Cách dùng `dict` để ánh xạ ký tự → giá trị.
- Kỹ thuật **duyệt chuỗi + nhìn phần tử kế tiếp**.
- Kỹ thuật **duyệt từ phải sang trái**.
- Cách tránh các lỗi rất hay gặp.
- Độ phức tạp thời gian và bộ nhớ.
- Cách tự viết lời giải chuẩn LeetCode.
- Bài tập vận dụng từ dễ đến nâng nhẹ.


## 1. Bảng giá trị cơ bản của số La Mã

Có 7 ký tự chính:

| Ký tự | Giá trị |
|---|---:|
| `I` | 1 |
| `V` | 5 |
| `X` | 10 |
| `L` | 50 |
| `C` | 100 |
| `D` | 500 |
| `M` | 1000 |

Ví dụ:

- `III = 1 + 1 + 1 = 3`
- `VIII = 5 + 1 + 1 + 1 = 8`
- `XV = 10 + 5 = 15`

Nhưng chưa phải lúc nào cũng chỉ cộng.


In [1]:
roman = {
    "I": 1,
    "V": 5,
    "X": 10,
    "L": 50,
    "C": 100,
    "D": 500,
    "M": 1000,
}

print(roman["I"])
print(roman["X"])
print(roman["M"])

1
10
1000


## 2. `dict` trong bài này dùng để làm gì?

Ta có:

```python
roman = {
    "I": 1,
    "V": 5,
    ...
}
```

Đây là một **dictionary**.

Có thể hiểu đơn giản:

> `dict` là một bảng tra cứu theo dạng **key → value**.

Trong bài này:

- key = ký tự La Mã
- value = giá trị số nguyên

Ví dụ:

```python
roman["X"]   # 10
roman["C"]   # 100
```

### Vì sao dùng `dict`?

Nếu không dùng `dict`, ta có thể phải viết:

```python
if ch == "I":
    value = 1
elif ch == "V":
    value = 5
elif ch == "X":
    value = 10
...
```

Dùng `dict` ngắn gọn hơn và đúng bản chất của bài toán: **tra giá trị của từng ký tự**.


# 3. Quy tắc đặc biệt: khi nào phải trừ?

Một số trường hợp số nhỏ đứng **trước** số lớn thì không cộng, mà phải trừ.

Các cặp đặc biệt thường gặp:

| Cặp | Giá trị |
|---|---:|
| `IV` | 4 |
| `IX` | 9 |
| `XL` | 40 |
| `XC` | 90 |
| `CD` | 400 |
| `CM` | 900 |

Ví dụ:

```text
IV
I = 1
V = 5

1 đứng trước 5
→ 5 - 1 = 4
```

Tương tự:

```text
IX = 10 - 1 = 9
XL = 50 - 10 = 40
CM = 1000 - 100 = 900
```

## Quan sát quan trọng

Nếu:

```text
giá trị hiện tại < giá trị kế tiếp
```

thì ta **trừ giá trị hiện tại**.

Ngược lại, ta **cộng**.

Đây chính là chìa khóa của lời giải.


# 4. Ví dụ phân tích bằng tay

## Ví dụ 1: `III`

```text
I  I  I
1  1  1
```

Không có số nhỏ đứng trước số lớn.

```text
1 + 1 + 1 = 3
```

## Ví dụ 2: `IV`

```text
I  V
1  5
```

Vì:

```text
1 < 5
```

nên:

```text
-1 + 5 = 4
```

## Ví dụ 3: `MCMXCIV`

Ta nhìn từng vị trí:

```text
M    C    M    X    C    I    V
1000 100 1000 10   100  1    5
```

- `M`: 1000 > 100 → cộng 1000
- `C`: 100 < 1000 → trừ 100
- `M`: 1000 > 10 → cộng 1000
- `X`: 10 < 100 → trừ 10
- `C`: 100 > 1 → cộng 100
- `I`: 1 < 5 → trừ 1
- `V`: ký tự cuối → cộng 5

Tổng:

```text
1000 - 100 + 1000 - 10 + 100 - 1 + 5
= 1994
```


# 5. Kỹ thuật 1 — Duyệt trái → phải và nhìn ký tự kế tiếp

Đây là cách rất tự nhiên cho người mới.

Ta duyệt từng ký tự `s[i]`.

Tại mỗi vị trí:

```python
current = roman[s[i]]
next_value = roman[s[i + 1]]
```

Nếu:

```python
current < next_value
```

→ trừ `current`.

Ngược lại:

→ cộng `current`.

### Vấn đề

Ở ký tự cuối cùng, không còn `s[i + 1]`.

Vì vậy ta chỉ duyệt đến:

```python
len(s) - 1
```

Sau vòng lặp, cộng ký tự cuối vào kết quả.


In [ ]:
def roman_to_int_left_to_right(s: str) -> int:
    roman = {
        "I": 1,
        "V": 5,
        "X": 10,
        "L": 50,
        "C": 100,
        "D": 500,
        "M": 1000,
    }

    total = 0

    for i in range(len(s) - 1):
        current = roman[s[i]]
        next_value = roman[s[i + 1]]

        if current < next_value:
            total -= current
        else:
            total += current

    total += roman[s[-1]]

    return total


print(roman_to_int_left_to_right("III"))
print(roman_to_int_left_to_right("IV"))
print(roman_to_int_left_to_right("MCMXCIV"))

## 6. Mổ xẻ vòng lặp

Dòng:

```python
for i in range(len(s) - 1):
```

nghĩa là:

- `i` chạy qua các chỉ số từ `0`
- đến trước `len(s) - 1`
- tức là **không chạy vào ký tự cuối**

Ví dụ:

```python
s = "MCM"
```

thì:

```text
len(s) = 3
range(len(s) - 1) = range(2)
```

nên `i` nhận:

```text
0, 1
```

Ta có thể an toàn dùng:

```python
s[i + 1]
```

vì:

- khi `i = 0` → xem `s[1]`
- khi `i = 1` → xem `s[2]`

Không bị vượt chỉ số.


In [ ]:
s = "MCM"

for i in range(len(s) - 1):
    print(
        "i =", i,
        "| hiện tại =", s[i],
        "| kế tiếp =", s[i + 1]
    )

# 7. Mô phỏng từng bước

Ta viết thêm `print` để quan sát cách thuật toán hoạt động.


In [ ]:
def trace_roman(s: str) -> int:
    roman = {
        "I": 1,
        "V": 5,
        "X": 10,
        "L": 50,
        "C": 100,
        "D": 500,
        "M": 1000,
    }

    total = 0

    for i in range(len(s) - 1):
        current = roman[s[i]]
        next_value = roman[s[i + 1]]

        if current < next_value:
            total -= current
            print(f"{s[i]}: {current} < {next_value} -> trừ {current}, total = {total}")
        else:
            total += current
            print(f"{s[i]}: {current} >= {next_value} -> cộng {current}, total = {total}")

    total += roman[s[-1]]
    print(f"{s[-1]} là ký tự cuối -> cộng {roman[s[-1]]}, total = {total}")

    return total


trace_roman("MCMXCIV")

# 8. Lời giải LeetCode chuẩn

Ta đưa logic vào đúng class mà đề bài yêu cầu.


In [ ]:
class Solution:
    def romanToInt(self, s: str) -> int:
        roman = {
            "I": 1,
            "V": 5,
            "X": 10,
            "L": 50,
            "C": 100,
            "D": 500,
            "M": 1000,
        }

        total = 0

        for i in range(len(s) - 1):
            if roman[s[i]] < roman[s[i + 1]]:
                total -= roman[s[i]]
            else:
                total += roman[s[i]]

        total += roman[s[-1]]

        return total

In [ ]:
sol = Solution()

tests = [
    ("III", 3),
    ("LVIII", 58),
    ("MCMXCIV", 1994),
    ("IV", 4),
    ("IX", 9),
]

for s, expected in tests:
    result = sol.romanToInt(s)
    print(f"{s:8} -> {result:4} | expected = {expected} | {'PASS' if result == expected else 'FAIL'}")

# 9. Cách suy nghĩ ngắn gọn để nhớ khi đi thi / phỏng vấn

Chỉ cần nhớ câu này:

> **Nếu số hiện tại nhỏ hơn số ngay sau nó → trừ. Ngược lại → cộng.**

Ví dụ:

```text
X C
10 < 100
→ -10 + 100 = 90
```

Ví dụ:

```text
V I I I
5 > 1
→ cộng 5
```

Vậy ta không cần học thuộc riêng từng cặp `IV`, `IX`, `XL`, ... để viết thuật toán.

Ta chỉ cần nhận ra **quy luật so sánh hai ký tự liền nhau**.


# 10. Kỹ thuật 2 — Duyệt từ phải sang trái

Đây là một cách rất hay và gọn.

Ví dụ:

```text
M C M
```

Ta bắt đầu từ bên phải.

- `M = 1000`: cộng
- sang `C = 100`
- vì `100 < 1000` nên trừ 100
- sang `M = 1000`
- vì `1000 >= 100` nên cộng

Ta cần lưu giá trị đã thấy trước đó ở bên phải:

```python
prev
```

Quy tắc:

```python
if current < prev:
    total -= current
else:
    total += current
```


In [ ]:
def roman_to_int_right_to_left(s: str) -> int:
    roman = {
        "I": 1,
        "V": 5,
        "X": 10,
        "L": 50,
        "C": 100,
        "D": 500,
        "M": 1000,
    }

    total = 0
    prev = 0

    for ch in reversed(s):
        current = roman[ch]

        if current < prev:
            total -= current
        else:
            total += current

        prev = current

    return total


print(roman_to_int_right_to_left("III"))
print(roman_to_int_right_to_left("IV"))
print(roman_to_int_right_to_left("MCMXCIV"))

## 11. `reversed(s)` là gì?

Nếu:

```python
s = "MCM"
```

thì:

```python
reversed(s)
```

cho ta duyệt theo thứ tự:

```text
M
C
M
```

nhưng bắt đầu từ ký tự cuối của chuỗi gốc.

Ví dụ dễ thấy hơn:

```python
s = "IV"
```

Duyệt bình thường:

```text
I -> V
```

Duyệt bằng `reversed(s)`:

```text
V -> I
```

Điểm hay:

- không cần truy cập `s[i + 1]`
- không cần xử lý riêng ký tự cuối
- code có thể ngắn hơn


In [ ]:
s = "MCMXCIV"

print("Chuỗi gốc:", list(s))
print("Duyệt ngược:", list(reversed(s)))

# 12. So sánh hai cách

## Cách 1: trái → phải + lookahead

```python
if roman[s[i]] < roman[s[i + 1]]:
```

### Ưu điểm

- Rất trực quan.
- Sát với cách ta đọc đề.
- Dễ giải thích cho người mới.

### Nhược điểm

- Phải cẩn thận với `i + 1`.
- Phải xử lý ký tự cuối riêng.

---

## Cách 2: phải → trái

```python
if current < prev:
```

### Ưu điểm

- Code gọn.
- Không lo `i + 1` vượt mảng.
- Không cần xử lý ký tự cuối riêng.

### Nhược điểm

- Ban đầu hơi khó hiểu hơn.

### Khuyên dùng khi mới học

Hãy thành thạo **cách 1 trước**.

Sau khi hiểu bản chất, học thêm cách 2 để thấy cách tối giản thuật toán.


# 13. Mẹo quan trọng

## Mẹo 1 — Đừng viết riêng 6 trường hợp đặc biệt nếu không cần

Cách này vẫn đúng:

```python
special = {
    "IV": 4,
    "IX": 9,
    "XL": 40,
    "XC": 90,
    "CD": 400,
    "CM": 900,
}
```

Nhưng dễ làm code dài và nhiều nhánh.

Thay vì vậy, chỉ cần dùng quy tắc:

```python
current < next_value
```

---

## Mẹo 2 — Hãy nghĩ theo giá trị, không theo ký tự

Sai tư duy:

```text
Nếu gặp I thì...
Nếu gặp X thì...
```

Tư duy tốt hơn:

```text
Giá trị hiện tại có nhỏ hơn giá trị kế tiếp không?
```

---

## Mẹo 3 — Cẩn thận vị trí cuối

Nếu dùng:

```python
s[i + 1]
```

thì không được để `i` chạy đến chỉ số cuối.

---

## Mẹo 4 — Biến tên rõ ràng

Nên dùng:

```python
current
next_value
total
```

thay vì:

```python
a
b
x
```

khi đang học.

Code dài thêm một chút nhưng dễ hiểu hơn rất nhiều.


# 14. Những lỗi thường gặp

## Lỗi 1 — Luôn cộng tất cả ký tự

```python
for ch in s:
    total += roman[ch]
```

Với:

```text
IV
```

sẽ ra:

```text
1 + 5 = 6
```

nhưng đáp án đúng là 4.

---

## Lỗi 2 — Trừ cả số lớn

Sai:

```text
IV = 1 - 5
```

Đúng:

```text
IV = -1 + 5
```

Ta trừ **số nhỏ đứng trước**, không trừ số lớn.

---

## Lỗi 3 — IndexError

Ví dụ:

```python
for i in range(len(s)):
    if roman[s[i]] < roman[s[i + 1]]:
```

Khi `i` là vị trí cuối:

```python
s[i + 1]
```

không tồn tại.

---

## Lỗi 4 — Quên cộng ký tự cuối

Nếu chỉ duyệt:

```python
for i in range(len(s) - 1):
```

mà quên:

```python
total += roman[s[-1]]
```

thì kết quả sẽ thiếu ký tự cuối.


# 15. Độ phức tạp

Gọi:

```text
n = len(s)
```

Ta duyệt chuỗi đúng một lần.

## Thời gian

```text
O(n)
```

## Bộ nhớ phụ

Dictionary chỉ có đúng 7 phần tử cố định.

Do đó:

```text
O(1)
```

### Kết luận

Lời giải rất hiệu quả:

```text
Time:  O(n)
Space: O(1)
```


# 16. Bài tập 1 — Tính bằng tay

Không chạy code trước.

Hãy tính:

1. `III`
2. `VI`
3. `IV`
4. `IX`
5. `XIV`
6. `XL`
7. `XC`
8. `CD`
9. `CM`
10. `MCMXCIV`

### Câu hỏi cần tự hỏi ở mỗi vị trí

> Giá trị hiện tại có nhỏ hơn giá trị đứng ngay sau không?


# 17. Bài tập 2 — Hoàn thiện code

Điền vào các chỗ `???`.


In [ ]:
def exercise_roman_to_int(s: str) -> int:
    roman = {
        "I": 1,
        "V": 5,
        "X": 10,
        "L": 50,
        "C": 100,
        "D": 500,
        "M": 1000,
    }

    total = 0

    # TODO:
    # for i in range(???):
    #     if ???:
    #         total -= ???
    #     else:
    #         total += ???
    #
    # total += ???
    #
    # return total

    raise NotImplementedError("Hãy tự hoàn thiện hàm này trước khi xem lời giải.")

## Gợi ý bài tập 2

Gợi ý 1:

```python
range(len(s) - 1)
```

Gợi ý 2:

So sánh:

```python
roman[s[i]]
```

với:

```python
roman[s[i + 1]]
```

Gợi ý 3:

Sau vòng lặp, đừng quên ký tự:

```python
s[-1]
```


# 18. Bài tập 3 — Viết phiên bản duyệt từ phải sang trái

Yêu cầu:

- dùng `reversed(s)`
- có biến `prev`
- nếu `current < prev` thì trừ
- ngược lại cộng


In [ ]:
def exercise_right_to_left(s: str) -> int:
    roman = {
        "I": 1,
        "V": 5,
        "X": 10,
        "L": 50,
        "C": 100,
        "D": 500,
        "M": 1000,
    }

    # TODO: tự viết
    raise NotImplementedError

# 19. Bài tập 4 — Dự đoán kết quả trước khi chạy

Hãy đoán kết quả:

```python
romanToInt("XXVII")
romanToInt("XLII")
romanToInt("XCIX")
romanToInt("CDXLIV")
romanToInt("MMXXVI")
```

Sau đó dùng cell dưới để kiểm tra.


In [ ]:
sol = Solution()

for s in ["XXVII", "XLII", "XCIX", "CDXLIV", "MMXXVI"]:
    print(s, "->", sol.romanToInt(s))

# 20. Bài tập 5 — Trace thuật toán

Với chuỗi:

```text
XLIV
```

Hãy tự lập bảng:

| i | s[i] | giá trị hiện tại | ký tự sau | giá trị sau | cộng / trừ | total |
|---|---|---:|---|---:|---|---:|

Không chạy code trước.

Sau đó kiểm tra bằng:


In [ ]:
trace_roman("XLIV")

# 21. Lời giải tham khảo cho bài tập 2

Chỉ xem sau khi đã tự thử.


In [ ]:
def exercise_roman_to_int_solution(s: str) -> int:
    roman = {
        "I": 1,
        "V": 5,
        "X": 10,
        "L": 50,
        "C": 100,
        "D": 500,
        "M": 1000,
    }

    total = 0

    for i in range(len(s) - 1):
        if roman[s[i]] < roman[s[i + 1]]:
            total -= roman[s[i]]
        else:
            total += roman[s[i]]

    total += roman[s[-1]]

    return total

# 22. Lời giải tham khảo cho bài tập 3


In [ ]:
def exercise_right_to_left_solution(s: str) -> int:
    roman = {
        "I": 1,
        "V": 5,
        "X": 10,
        "L": 50,
        "C": 100,
        "D": 500,
        "M": 1000,
    }

    total = 0
    prev = 0

    for ch in reversed(s):
        current = roman[ch]

        if current < prev:
            total -= current
        else:
            total += current

        prev = current

    return total

# 23. Mini challenge

Hãy sửa lời giải sao cho dictionary nằm ngoài class:

```python
ROMAN = {
    ...
}
```

Sau đó trong hàm chỉ dùng:

```python
ROMAN[s[i]]
```

### Câu hỏi tư duy

1. Vì sao dictionary này có thể xem là `O(1)` bộ nhớ?
2. Vì sao bài này không cần `Counter`?
3. Vì sao `set` không phù hợp để lưu bảng giá trị số La Mã?
4. `dict` giải quyết đúng nhu cầu nào của bài toán?

Gợi ý câu 3:

Ta cần:

```text
"I" -> 1
"V" -> 5
```

Trong khi `set` chủ yếu lưu **các phần tử duy nhất**, không lưu ánh xạ `key -> value`.


# 24. Tóm tắt kiến thức cần nhớ

## Kiến thức Python

```python
dict
range()
len()
s[i]
s[i + 1]
s[-1]
reversed()
```

## Kỹ thuật thuật toán

### Kỹ thuật 1

**Lookahead** — nhìn phần tử kế tiếp.

```python
if current < next_value:
    trừ
else:
    cộng
```

### Kỹ thuật 2

**Duyệt từ phải sang trái** và nhớ giá trị trước đó.

```python
if current < prev:
    trừ
else:
    cộng
```

## Công thức nhớ nhanh nhất

> **Nhỏ đứng trước lớn → trừ. Còn lại → cộng.**

## Độ phức tạp

```text
Time:  O(n)
Space: O(1)
```


# 25. Phiên bản cuối cùng nên nhớ

```python
class Solution:
    def romanToInt(self, s: str) -> int:
        roman = {
            "I": 1,
            "V": 5,
            "X": 10,
            "L": 50,
            "C": 100,
            "D": 500,
            "M": 1000,
        }

        total = 0

        for i in range(len(s) - 1):
            if roman[s[i]] < roman[s[i + 1]]:
                total -= roman[s[i]]
            else:
                total += roman[s[i]]

        total += roman[s[-1]]

        return total
```

Khi đã hiểu phiên bản này, bạn gần như đã nắm được toàn bộ bản chất của bài **Roman to Integer**.
